# Text to 3D AI Model - Colab + ngrok

This notebook runs the GitHub project on a Colab GPU and exposes the FastAPI frontend/backend through ngrok.

## 1. Clone the GitHub repository

In [2]:
REPO_URL = "https://github.com/LucyAlex12/Text_to_3d_ai_model.git"

!rm -rf Text_to_3d_ai_model
!git clone {REPO_URL}
%cd Text_to_3d_ai_model

Cloning into 'Text_to_3d_ai_model'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 74 (delta 22), reused 66 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 30.27 MiB | 10.74 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/Text_to_3d_ai_model/Text_to_3d_ai_model


## 2. Install dependencies

If Colab asks you to restart the runtime after installs, restart it, then rerun the cells from the top.

In [3]:
# Clean conflicting packages first
!pip uninstall -y cupy cupy-cuda12x cupy-cuda11x numpy pymatting rembg

# Stable numpy for Colab + torch ecosystem
!pip install numpy==1.26.4

# Install backend dependencies
!pip install rembg==2.0.59 pymatting==1.1.12
!pip install fastapi uvicorn python-multipart pyngrok pillow
!pip install transformers diffusers accelerate safetensors
!pip install trimesh xatlas pygltflib

Found existing installation: cupy-cuda12x 14.0.1
Uninstalling cupy-cuda12x-14.0.1:
  Successfully uninstalled cupy-cuda12x-14.0.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: PyMatting 1.1.15
Uninstalling PyMatting-1.1.15:
  Successfully uninstalled PyMatting-1.1.15
Found existing installation: rembg 2.0.69
Uninstalling rembg-2.0.69:
  Successfully uninstalled rembg-2.0.69
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, which is not installed.
pylibcugraph-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, which is not installed.
cudf-

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00


## 3. Download TripoSR weights

`TripoSR/model.ckpt` is large, so the notebook downloads it from Hugging Face instead of storing it in Git.

In [3]:
from pathlib import Path
from huggingface_hub import hf_hub_download

Path("TripoSR").mkdir(exist_ok=True)

for filename in ["config.yaml", "model.ckpt"]:
    hf_hub_download(
        repo_id="stabilityai/TripoSR",
        filename=filename,
        local_dir="TripoSR"
    )

print("TripoSR checkpoint ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.yaml:   0%|          | 0.00/987 [00:00<?, ?B/s]

model.ckpt:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

TripoSR checkpoint ready


## 4. Configure ngrok

Create an ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken.

The next cell uses `getpass()` instead of Colab Secrets because Colab's secret vault can fail with `Failed to fetch` / `await connected: disconnected` errors.

Optional: if you reserved an ngrok static domain, enter it when prompted, for example `your-name.ngrok-free.app`.

In [4]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ").strip()
NGROK_STATIC_DOMAIN = input("Optional ngrok static domain, or press Enter: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("ngrok authtoken is required")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

if NGROK_STATIC_DOMAIN:
    print("ngrok configured with static domain:", NGROK_STATIC_DOMAIN)
else:
    print("ngrok configured with a temporary URL")

Paste your ngrok authtoken: ··········
Optional ngrok static domain, or press Enter: 
ngrok configured with a temporary URL


## 5. Start backend and expose the app

Open the printed ngrok URL in a normal browser tab. Do not use Colab's iframe preview.

In [7]:
import os
import subprocess
import time
import requests
from pyngrok import ngrok

os.environ["IMAGE_MODEL_KIND"] = "sd15"
os.environ["SDXL_WIDTH"] = "512"
os.environ["SDXL_HEIGHT"] = "512"
os.environ["SDXL_STEPS"] = "20"
os.environ["SDXL_GUIDANCE_SCALE"] = "7.0"
os.environ["TRIPOSR_MAX_MC_RESOLUTION"] = "256"

ngrok.kill()

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting backend. This can take a few minutes while models load...")

for _ in range(240):
    if server.poll() is not None:
        remaining = server.stdout.read() if server.stdout else ""
        raise RuntimeError("Backend stopped before it was ready. Logs:\n" + remaining[-4000:])
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("Backend is ready")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not become ready. Run the logs cell below.")

if NGROK_STATIC_DOMAIN:
    tunnel = ngrok.connect(8000, "http", domain=NGROK_STATIC_DOMAIN)
else:
    tunnel = ngrok.connect(8000, "http")

public_url = tunnel.public_url

print("\nOpen this public app URL:")
print(public_url)
print("\nKeep this Colab runtime running while using the app.")
print("If ngrok shows a browser warning page, click through once. The app also sends the ngrok-skip-browser-warning header for API/model requests.")

Starting backend. This can take a few minutes while models load...


RuntimeError: Backend stopped before it was ready. Logs:
^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 853, in invoke
    return callback(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 441, in main
    run(
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 617, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvloo

## Optional: view backend logs

Run this only if the app fails or you want to monitor generation logs.

In [ ]:
while True:
    line = server.stdout.readline()
    if not line:
        break
    print(line, end="")